# Extreme values

The `surprises` pass reports numeric values that sit apart from the rest
of their column — far enough that the distance itself is the finding. It
runs at the standard and deep budgets and emits `intervals` findings of type
`extreme_value` (form `isolation_gap`): the extreme values and their row
indices, the edge of the remaining bulk, and the size of the gap between
them, stated as a multiple of the bulk's central spread.

The pass exists because a column card does not show this. A maximum is
one number among the summary statistics whether it sits just past the
bulk or a full spread away from it, and a mean or a standard deviation
absorbs the extreme into the quantity meant to expose it.

The same pass runs a second scan over categorical columns. It looks
for the mirror image of an extreme value: not a number far from its
column's bulk, but a level that appears inside a context where
almost every row carries a different one. Those findings are type
`categorical_outlier`; the test is described under
*Pattern-breaking categorical values* below.


## The isolation test

Each tail of a numeric column is tested separately. The candidates are
the values past the Tukey fence (Q3 + 1.5 · IQR at the top, Q1 −
1.5 · IQR at the bottom). Candidates are tested as a cluster of one to
three values, not one at a time: the cluster is isolated when the gap
between its nearest member and the edge of the remaining bulk is at
least 1.0 × the bulk's central spread (p90 − p10 of the remainder).

Testing the cluster rather than each value matters. Two adjacent
extremes have a near-zero gap between them, so a test that measures
each value's distance to its nearest neighbour sees the pair as
ordinary company for each other and reports neither. Measured against
the bulk instead, the pair's joint gap is what it is.

## The second scale

For a column whose values are all positive, isolation must hold on two
scales: the raw scale above, and the log scale at half the bar (a log
gap of at least 0.5 × the bulk's log-scale central spread). A
right-skewed positive column — a lab measurement, a count, an amount —
has a long raw-scale tail by construction; distances that look isolated
in raw units are the distribution's ordinary shape in ratio units. The
log test is what separates that shape from a value that is extreme
relative to the column's own proportions.

## Extremes within a subset

A value can be unremarkable in its column and isolated inside one
category of it. The pass therefore repeats the test within each level
of up to eight eligible conditioning columns per target, with 2 to 32 levels, for scopes holding at
least 30 target values. A finding from a subset names the condition in
its claim. Values already reported by the column-wide test are not
reported again per subset.

The example below builds all three cases into one frame: a pair of
adjacent extremes in `age` (148 and 151 against a bulk that ends at
82), two values (79 and 82) that are ordinary in the column but
isolated inside `cohort_b = true`, whose other ages stop at 45, and a
lognormal `lab_value` whose raw tail — maximum 208.8 against a median
near 20 — produces no finding under the log-scale test.

In [ ]:
import random

import polars as pl

import onceover

ages, cohort = [], []
for i in range(600):
    if i % 4 == 0:
        cohort.append(True)
        ages.append(20 + (i // 4) % 26)        # 20..45 within cohort_b
    else:
        cohort.append(False)
        ages.append(20 + i % 41)               # 20..60 outside it
ages[301] = 79
cohort[301] = True                             # subset extremes
ages[302] = 82
cohort[302] = True
ages[500] = 148
ages[501] = 151                                # the adjacent pair

rng = random.Random(11)
lab = [round(rng.lognormvariate(3.0, 0.9), 2) for _ in range(600)]

df = pl.DataFrame({"age": ages, "cohort_b": cohort, "lab_value": lab})
report = onceover.profile(df)

for f in report.findings:
    if f["specification"].get("form") == "isolation_gap":
        print(f["claim"])
        print("   evidence:", {k: f["evidence"][0]["metrics"][k]
              for k in ("bulk_edge", "gap", "gap_ratio", "extreme_rows")})
print("lab_value maximum:", df["lab_value"].max(),
      "- no extreme_value finding for it")

## Pattern-breaking categorical values

A categorical value has no distance from a bulk, so the isolation
test does not transfer. The categorical scan instead fixes a
context — a conjunction of levels from other columns, such as
`stem_cankers = above-sec-nde` — and asks whether one level of the
target column is near-universal inside it. A context qualifies
when it holds at least 30 rows, one level accounts for at least
99% of them, and at most 3 rows carry a different level. Those
rows are the finding.

Contexts run to three conditions. Single levels of every eligible
column are tested first. Deeper contexts are built from the 12
condition columns whose single levels come closest to the bar for
that target: pairs among their levels, and triples formed by
extending pairs whose dominant share is already at least 0.95.
Candidates are ranked by the binomial probability of the dominant
count under the dominant level's rate in the full column, so a
context that merely restates the column's overall distribution
ranks below one where the level is ordinary globally and
near-universal locally. At most 3 findings are reported per
target column; a context is reported only when it adds a breaking row, and it lists every row breaking its pattern.

The example below plants one breaking row in a frame built from
four 100-row blocks. In the first block (`c1 = a`, `c2 = b`),
`outcome` is `y` in every row but one: row 42 carries `x`. No
single condition separates anything — `c1 = a` and `c2 = b` each
hold `outcome = x` in about half their rows. The same pattern
also reads in the other directions: within `c2 = b` and
`outcome = x`, `c1` is `w` except at row 42. The scan reports
the readings as one finding with an evidence record for each,
because the same row breaks all of them.


In [ ]:
c1, c2, outcome = [], [], []
blocks = [("a", "b", "y"), ("a", "z", "x"), ("w", "b", "x"), ("w", "z", None)]
for x1, x2, val in blocks:
    for i in range(100):
        c1.append(x1)
        c2.append(x2)
        outcome.append(val if val is not None else ("x" if i % 2 == 0 else "y"))
outcome[42] = "x"          # the one breaker of the (a, b) pattern

df = pl.DataFrame({"c1": c1, "c2": c2, "outcome": outcome})
report = onceover.profile(df)

for f in report.findings:
    if f.get("type") == "categorical_outlier":
        print(f["claim"])
        for e in f["evidence"]:
            print("   evidence:", e["metrics"])


## Reading the evidence

The evidence record carries the test's quantities, so the claim can be
checked without re-running it: `bulk_edge` (the largest bulk value on
the tested side), `gap` and `gap_ratio` (the distance to the nearest
extreme, and that distance divided by the bulk's central spread),
`central_spread_p90_p10`, `extreme_values` with `extreme_rows` (their
0-based row indices), `side`, and `n`, the number of values in the
scope tested.

A `categorical_outlier` finding's evidence carries the context's
quantities: `n` (rows in the context), `dominant_level` and
`dominant_share`, `outlier_count`, `outlier_rows` with
`outlier_values`, and `log10_significance`, the ranking
statistic. The claim states one framing of the pattern; the
finding's subject names every column involved, and each further
framing the scan found appears as its own evidence record.

The pass has a shared limit of 20 findings. Every numeric column is
scanned — the scans run in parallel — and the limit is applied
afterwards in column order: whole-column findings, ranked by
strength, take the budget first, subset findings share what
remains, and the categorical scan runs only while budget is left.
The result does not depend on the thread count. Within a tested
scope the scan uses all eligible rows.

## What the pass does not report

- **A value that is surprising only against a prediction.** A result
  that is ordinary for its column but wrong for its row — high given
  the patient's other measurements, low given the account's history —
  is surprising relative to a model of the other columns, not relative
  to the bulk. No finding in this pass speaks to that form; the two-column case is the row-consistency check at the end of this page.
- **Clusters of more than three extremes.** A separated group of four
  or more values is a second mode or a merged subpopulation, not an
  extreme of this bulk; other passes describe it.
- **Small scopes.** A subset or context with fewer than 30 values is
  not tested: its spread or share estimate would rest on the same
  handful of values being judged.
- **Categorical contexts past the search limits.** The categorical
  scan tests text and boolean targets, contexts of at most three
  conditions, and — below single conditions — only the 12 condition
  columns whose single levels come closest to the bar. A pattern
  that needs a fourth condition, or a first condition far from the
  bar, is out of reach. The unconditioned case is not reported
  either: a column that is 99% one level overall is a concentration
  finding, not a row-level surprise.

## Rows that disagree with a paired column

The checks above judge a value against its own column's bulk. Two checks in the `row_consistency` detector judge a row against one other column. Both report `intervals` findings of type `row_inconsistency`, told apart by the `kind` field in the evidence. The detector is in the default set and runs after the dependence screen, whose pair statistics it reuses. Its third check, near-duplicate rows, is covered in [keys and duplicates](keys_duplicates.ipynb).

**Rows off a fitted line** (`kind: residual_outlier`). For each numeric pair the dependence screen found tightly correlated (|r| ≥ 0.9; at most 256 pairs, strongest first, scaffolding columns excluded), the detector fits a line over the pair's complete rows \u2014 at least 100 are required \u2014 then refits after trimming the points beyond three robust scales, so the outliers being looked for cannot pull the line towards themselves. Residuals are scored against 1.4826 × the median absolute residual. A finding needs at least one row at 8 or more robust units from the refitted line, and the outlying rows must stay isolated: at most max(5, 0.5% of the pair's rows), with the largest ordinary residual below 60% of the smallest flagged one, so a fat-tailed bulk produces no finding. An exact relation is left to the identity checks: when the residual scale is effectively zero, the pair is skipped. The evidence carries the fitted `slope` and `intercept`, the count of `outliers`, the first 20 outlying `rows`, worst first, and `ordinary_band`, the largest residual among the rows that were not flagged.

**Swapped values** (`kind: swapped_values`). The scan runs over the columns the frame model roled as measures. A column's usual range is its middle 90% \u2014 the 5th to 95th percentile over a sample of at most 20,000 rows \u2014 widened by half its width on each side; a column needs at least 100 sampled values to take part. Pairs whose widened ranges do not overlap (at most 5,000 such pairs) are candidates: a row counts as swapped when each column's value falls inside the other's usual range. A pair with no swap in the sample is not scanned in full; the counts in a finding come from the full scan. A finding needs at least 2 swapped rows and at most 5% of the pair's complete rows, and the two moves must coincide: the swapped rows must be at least 80% of the rows where either value moved into the other's range. One column's tail value can land in another column's range by chance; the other column then stays in its own range, and the coincidence test fails. The evidence carries `swapped_rows`, the first 20 `rows`, both usual ranges (`range_first`, `range_second`), and the complete-row count.


In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 400
x = rng.uniform(0, 50, n)
y = 2.0 * x + 1.0 + rng.normal(0, 0.5, n)
y[[10, 200, 399]] += 25.0  # three rows knocked off the line
a = rng.normal(100, 5, n)
b = rng.normal(200, 5, n)
swap = [3, 100, 201, 302]
a[swap], b[swap] = b[swap].copy(), a[swap].copy()  # four rows hold each other's values

report = onceover.profile(pl.DataFrame({"x": x, "y": y, "a": a, "b": b}))

for f in report.findings:
    if f.get("type") == "row_inconsistency":
        print(f["claim"])
        for e in f["evidence"]:
            print("   evidence:", e["metrics"])

by_kind = {
    f["evidence"][0]["metrics"]["kind"]: f
    for f in report.findings
    if f.get("type") == "row_inconsistency"
}
assert by_kind["residual_outlier"]["evidence"][0]["metrics"]["rows"] == [10, 200, 399]
assert by_kind["swapped_values"]["evidence"][0]["metrics"]["rows"] == swap


Both planted defects are found, and the evidence names the planted rows exactly. Rows 10, 200, and 399 sit 25 units above the line the other 397 rows follow; the claim reports the fitted line, and `ordinary_band` (1.36 here) is how far the unflagged rows stay from it. Rows 3, 100, 201, and 302 hold each other's values, each inside the other's usual range as the evidence records them. Every value in the table is ordinary for its own column. Only the pairings are wrong, which is why a column-at-a-time scan cannot see either defect.
